# Commentary Ingestion Pipeline
## Part 1 — YouTube Transcript Ingestion

Searches YouTube for race highlight videos and saves transcripts to disk.
**No Claude calls are made in Part 1.**

Run Part 1 daily (up to ~90 races per day within free quota limits).
The pipeline is fully resumable — already-cached races are skipped automatically.

---

## Part 2 — Claude Tactical Extraction

Reads saved transcripts and uses Claude to extract structured tactical events.
**Only processes transcripts not yet extracted — never re-charges for existing work.**

Run Part 2 independently after Part 1, at your own pace.

---

### Directory structure
```
data/commentary/
  raw/          ← Part 1 output: one JSON per race with transcript
  extracted/    ← Part 2 output: one JSON per race with tactical events
```

In [2]:
import os
import re
import json
import time
import datetime
from pathlib import Path
from dotenv import load_dotenv

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled,
    VideoUnavailable,
)
from rapidfuzz import fuzz
import pandas as pd
import anthropic

load_dotenv()

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
ANTHROPIC_KEY   = os.getenv("ANTHROPIC_API_KEY")
PROCESSED_DATA  = Path("../data/processed")
RAW_DIR         = Path("../data/commentary/raw")
EXTRACTED_DIR   = Path("../data/commentary/extracted")
RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

ytt     = YouTubeTranscriptApi()
youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
claude  = anthropic.Anthropic()

print(f"YouTube API key: {'SET' if YOUTUBE_API_KEY else 'MISSING'}")
print(f"Anthropic key:   {'SET' if ANTHROPIC_KEY else 'MISSING'}")
print(f"Raw dir:         {RAW_DIR}")
print(f"Extracted dir:   {EXTRACTED_DIR}")
print(f"\nExisting raw transcripts:  {len(list(RAW_DIR.glob('*.json')))}")
print(f"Existing extractions:      {len(list(EXTRACTED_DIR.glob('*.json')))}")


YouTube API key: SET
Anthropic key:   SET
Raw dir:         ..\data\commentary\raw
Extracted dir:   ..\data\commentary\extracted

Existing raw transcripts:  63
Existing extractions:      0


In [3]:
CHANNEL_REGISTRY = [
    {
        "id":       "UCqZQlzSHbVJrwrn5XvzrzcA",
        "name":     "NBC Sports",
        "coverage": "Grand Tours, Monuments, WorldTour",
    },
    {
        "id":       "UCu7phdCr-raU7OaJfEpHZww",
        "name":     "GCN Racing",
        "coverage": "All WorldTour",
    },
    {
        "id":       "UCfDfvvMARk4TKcC62ALi6eA",
        "name":     "TNT Sports Cycling",
        "coverage": "Grand Tours, European classics",
    },
    {
        "id":       "UCuTaETsuCOkJ0H_GAztWt0Q",
        "name":     "GCN",
        "coverage": "All WorldTour",
    },
]


In [4]:
# ── Part 1 Functions: Search, Score, Find ─────────────────

def search_channel(
    race_name: str,
    channel_id: str,
    race_date: str,
    stage: int = None,
    max_results: int = 3
) -> list[dict]:
    race_date = str(race_date)[:10]
    race_dt   = datetime.datetime.strptime(race_date, "%Y-%m-%d")

    if stage:
        query = f"{race_name} {race_date[:4]} stage {stage} highlights"
    else:
        query = f"{race_name} {race_date[:4]} highlights"

    published_after  = (race_dt - datetime.timedelta(days=3)).strftime("%Y-%m-%dT00:00:00Z")
    published_before = (race_dt + datetime.timedelta(days=30)).strftime("%Y-%m-%dT00:00:00Z")

    try:
        response = youtube.search().list(
            part="snippet",
            channelId=channel_id,
            q=query,
            type="video",
            maxResults=max_results,
            order="relevance",
            publishedAfter=published_after,
            publishedBefore=published_before,
        ).execute()
        results = []
        for item in response.get("items", []):
            results.append({
                "video_id":    item["id"]["videoId"],
                "title":       item["snippet"]["title"],
                "description": item["snippet"]["description"][:300],
                "published":   item["snippet"]["publishedAt"],
                "url":         f"https://youtube.com/watch?v={item['id']['videoId']}",
                "channel_id":  channel_id,
                "channel":     next(
                    (c["name"] for c in CHANNEL_REGISTRY if c["id"] == channel_id),
                    "unknown"
                ),
            })
        return results
    except HttpError as e:
        if "quotaExceeded" in str(e):
            print("Daily quota limit reached — resets at midnight Pacific")
            raise
        print(f"  Search error: {e}")
        return []


def score_video(video: dict, race_name: str, race_date: str, stage: int = None) -> float:
    score    = 0.0
    title    = video["title"].lower()
    pub_str  = video["published"][:10]
    race_date = str(race_date)[:10]
    race_dt   = datetime.datetime.strptime(race_date, "%Y-%m-%d")
    race_year = race_dt.year
    try:
        pub_dt = datetime.datetime.strptime(pub_str, "%Y-%m-%d")
    except ValueError:
        return -999.0

    days_diff = abs((pub_dt - race_dt).days)
    if days_diff == 0:       score += 60
    elif days_diff <= 1:     score += 50
    elif days_diff <= 3:     score += 35
    elif days_diff <= 7:     score += 15
    elif days_diff > 30:
        score -= 40 * abs(pub_dt.year - race_year)

    name_score = fuzz.partial_ratio(race_name.lower(), title)
    score += name_score * 0.25
    if name_score < 40:
        score -= 40

    if stage:
        patterns = [f"stage {stage}", f"stage{stage}", f"étape {stage}",
                    f"tappa {stage}", f"etapa {stage}"]
        score += 30 if any(p in title for p in patterns) else -10

    if str(race_year) in title:                                    score += 15
    if "extended" in title:                                        score += 10
    elif "highlights" in title:                                    score += 5
    if "nbc" in video.get("channel", "").lower() and "extended" in title: score += 5

    skip = ["preview", "interview", "training", "behind the scenes",
            "full season", "beginner", "guide", "how to", "shorts",
            "power data", "vs col"]
    if any(kw in title for kw in skip):                            score -= 20

    womens = ["femmes", "feminine", "women", "ladies"]
    if any(kw in title for kw in womens):                          score -= 30

    return round(score, 2)


def find_best_video(
    race_name: str,
    race_date: str,
    stage: int = None,
    confidence_threshold: float = 100.0,
    verbose: bool = True
) -> dict:
    all_candidates = []
    channels_used  = 0

    for channel in CHANNEL_REGISTRY:
        if verbose:
            print(f"  Searching {channel['name']}...")
        results = search_channel(race_name, channel["id"], race_date, stage)
        channels_used += 1
        for video in results:
            video["match_score"] = score_video(video, race_name, race_date, stage)
        all_candidates.extend(results)
        if all_candidates:
            best = max(all_candidates, key=lambda x: x["match_score"])
            if best["match_score"] >= confidence_threshold:
                if verbose:
                    print(f"  High-confidence match found — stopping early")
                break
        time.sleep(0.3)

    if not all_candidates:
        return {"found": False, "race_name": race_name,
                "race_date": race_date, "stage": stage,
                "quota_used": channels_used * 100}

    best = max(all_candidates, key=lambda x: x["match_score"])
    best["found"]          = True
    best["quota_used"]     = channels_used * 100
    best["all_candidates"] = all_candidates
    return best


In [5]:
def fetch_transcript(video_id: str) -> dict:
    try:
        transcript = ytt.fetch(video_id)
        raw_text   = " ".join([s.text for s in transcript])
        clean_text = re.sub(r'\[.*?\]', '', raw_text)
        clean_text = re.sub(r'\(.*?\)', '', clean_text)
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()
        duration_secs = round(transcript[-1].start, 0) if transcript else 0
        return {
            "success":       True,
            "video_id":      video_id,
            "snippet_count": len(transcript),
            "raw_chars":     len(raw_text),
            "clean_chars":   len(clean_text),
            "duration_secs": duration_secs,
            "duration_mins": round(duration_secs / 60, 1),
            "clean_text":    clean_text,
            "preview_start": clean_text[:500],
            "preview_end":   clean_text[-500:],
            "error":         None,
        }
    except NoTranscriptFound:
        return {"success": False, "video_id": video_id, "error": "no_transcript"}
    except TranscriptsDisabled:
        return {"success": False, "video_id": video_id, "error": "transcripts_disabled"}
    except VideoUnavailable:
        return {"success": False, "video_id": video_id, "error": "video_unavailable"}
    except Exception as e:
        return {"success": False, "video_id": video_id, "error": str(e)}


In [6]:
def process_race(
    race_name: str,
    race_date: str,
    stage: int = None,
    verbose: bool = True
) -> dict:
    """
    Full Part 1 pipeline for one race:
    1. Search YouTube across all channels
    2. Fetch transcript for best match
    3. Save raw JSON to data/commentary/raw/
    No Claude calls — transcript only.
    """
    label     = f"{str(race_date)[:4]} {race_name}"
    label    += f" Stage {stage}" if stage else ""
    safe_name = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
    out_path  = RAW_DIR / f"{safe_name}.json"

    if out_path.exists():
        if verbose:
            print(f"  Cache hit: {out_path.name}")
        with open(out_path, encoding="utf-8") as f:
            return json.load(f)

    result = {"label": label, "race_name": race_name, "race_date": str(race_date)[:10],
              "stage": stage, "video": None, "transcript": None, "status": "pending"}

    if verbose:
        print(f"\nProcessing: {label}")

    video = find_best_video(race_name, str(race_date)[:10], stage, verbose=verbose)

    if not video.get("found"):
        result["status"] = "no_video_found"
        if verbose:
            print(f"  ✗ No video found")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)
        return result

    result["video"] = {
        "video_id":       video["video_id"],
        "title":          video["title"],
        "url":            video["url"],
        "channel":        video["channel"],
        "published":      video["published"],
        "match_score":    video["match_score"],
        "all_candidates": [
            {"video_id": c["video_id"], "title": c["title"],
             "channel": c["channel"], "published": c["published"],
             "match_score": c["match_score"]}
            for c in video.get("all_candidates", [])
        ]
    }

    if verbose:
        print(f"  ✓ Video: [{video['match_score']:.0f}] {video['title'][:60]}")
        print(f"    Channel: {video['channel']} | Published: {video['published'][:10]}")
        print(f"  Fetching transcript...")

    transcript = fetch_transcript(video["video_id"])

    if not transcript["success"]:
        result["status"] = f"transcript_error:{transcript['error']}"
        if verbose:
            print(f"  ✗ Transcript error: {transcript['error']}")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)
        return result

    result["transcript"] = {
        "snippet_count": transcript["snippet_count"],
        "raw_chars":     transcript["raw_chars"],
        "clean_chars":   transcript["clean_chars"],
        "duration_mins": transcript["duration_mins"],
        "clean_text":    transcript["clean_text"],
        "preview_start": transcript["preview_start"],
        "preview_end":   transcript["preview_end"],
    }
    result["status"] = "transcript_saved"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    if verbose:
        print(f"  ✓ Saved: {transcript['clean_chars']:,} chars | "
              f"{transcript['duration_mins']:.1f} min → {out_path.name}")

    time.sleep(1)
    return result


print("process_race ready")


process_race ready


In [7]:
# ── Build race index from dataset ────────────────────────
merged_df = pd.read_csv(
    PROCESSED_DATA / "merged_uci_races.csv",
    low_memory=False
)
merged_df["Date"] = pd.to_datetime(merged_df["Date"])

race_index = (
    merged_df[["Race Name", "Race_results", "Date", "Year_results", "Stage_results"]]
    .drop_duplicates("Race Name")
    .sort_values("Date")
    .reset_index(drop=True)
)

already_done = len(list(RAW_DIR.glob("*.json")))
remaining    = len(race_index) - already_done

print(f"Total unique races:     {len(race_index):,}")
print(f"Already ingested:       {already_done:,}")
print(f"Remaining:              {remaining:,}")


Total unique races:     896
Already ingested:       63
Remaining:              833


In [33]:
def parse_race_name_and_stage(race_name_full: str) -> tuple[str, int]:
    clean = re.sub(r"^\d{4}\s+", "", race_name_full).strip()
    stage_match = re.search(r"Stage\s+(\d+)", clean, re.IGNORECASE)
    stage = int(stage_match.group(1)) if stage_match else None
    race  = re.sub(r"\s*Stage\s+\d+", "", clean, flags=re.IGNORECASE).strip()
    return race, stage


def run_ingestion(
    race_index: pd.DataFrame,
    max_races: int = None,
    verbose: bool = False
) -> dict:
    total     = len(race_index) if max_races is None else max_races
    processed = skipped = success = no_video = errors = 0
    no_video_list = []
    error_list    = []

    print(f"Starting ingestion run: up to {total} races")
    print(f"{'─'*55}")

    for i, row in race_index.iterrows():
        if processed >= total:
            break

        race_name_full = row["Race Name"]
        race_date      = str(row["Date"])[:10]
        race_name, stage = parse_race_name_and_stage(race_name_full)

        label     = f"{race_date[:4]} {race_name}" + (f" Stage {stage}" if stage else "")
        safe_name = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
        cache_path = RAW_DIR / f"{safe_name}.json"

        # Skip anything already cached — success, no_video, or error
        if cache_path.exists():
            skipped += 1
            continue

        if (processed - skipped) % 10 == 0:
            print(f"  [{processed:>3} seen | {success} saved | "
                  f"{no_video} no_video | {errors} errors]")

        try:
            result = process_race(race_name, race_date, stage, verbose=verbose)
            status = result["status"]

            if status == "transcript_saved":
                success += 1
            elif status == "no_video_found":
                no_video += 1
                no_video_list.append(label)
            else:
                errors += 1
                error_list.append((label, status))

        except HttpError as e:
            if "quotaExceeded" in str(e):
                print(f"\n⚠ Quota exhausted after {success} new transcripts.")
                print(f"  Re-run tomorrow — already-saved races will be skipped.")
                break
            errors += 1
            error_list.append((label, str(e)[:80]))
            error_record = {
                "label": label, "race_name": race_name,
                "race_date": race_date, "stage": stage,
                "status": f"exception:{str(e)[:100]}",
                "video": None, "transcript": None
            }
            with open(cache_path, "w") as f:
                json.dump(error_record, f, indent=2)

        except Exception as e:
            errors += 1
            error_list.append((label, str(e)[:80]))
            error_record = {
                "label": label, "race_name": race_name,
                "race_date": race_date, "stage": stage,
                "status": f"exception:{str(e)[:100]}",
                "video": None, "transcript": None
            }
            with open(cache_path, "w") as f:
                json.dump(error_record, f, indent=2)

        processed += 1
        time.sleep(45)

    print(f"\n{'─'*55}")
    print(f"Run complete")
    print(f"  Transcripts saved:   {success}")
    print(f"  No video found:      {no_video}")
    print(f"  Errors:              {errors}")
    print(f"  Skipped (cached):    {skipped}")
    print(f"  Total in raw dir:    {len(list(RAW_DIR.glob('*.json')))}")

    if no_video_list:
        print(f"\n── No video found ({len(no_video_list)}) ──")
        for label in no_video_list:
            print(f"  ✗ {label}")

    if error_list:
        print(f"\n── Errors ({len(error_list)}) ──")
        for label, reason in error_list:
            print(f"  ✗ {label}")
            print(f"    reason: {reason}")

    print(f"\n── To manually override a no_video result ──")
    print(f"  Delete the cached file and re-run, or use process_race() directly:")
    print(f"  e.g. process_race('Paris-Roubaix', '2023-04-09', stage=None, verbose=True)")

    return {
        "success":      success,
        "no_video":     no_video,
        "errors":       errors,
        "skipped":      skipped,
        "no_video_list": no_video_list,
        "error_list":   error_list,
    }


In [34]:
# ── PART 1: RUN THIS CELL DAILY ──────────────────────────
# Processes up to 90 races per run (safe daily quota limit)
# Already-cached races are skipped automatically
# Change max_races=None to process everything remaining in one go

stats = run_ingestion(
    race_index=race_index,
    max_races=90,       
    verbose=False,      # set True to see per-race detail
)

Starting ingestion run: up to 90 races
───────────────────────────────────────────────────────
  [  7 seen | 6 saved | 1 no_video | 0 errors]
  [ 17 seen | 16 saved | 1 no_video | 0 errors]
  [ 27 seen | 26 saved | 1 no_video | 0 errors]
  [ 37 seen | 29 saved | 8 no_video | 0 errors]
Daily quota limit reached — resets at midnight Pacific

⚠ Quota exhausted after 29 new transcripts.
  Re-run tomorrow — already-saved races will be skipped.

───────────────────────────────────────────────────────
Run complete
  Transcripts saved:   29
  No video found:      10
  Errors:              0
  Skipped (cached):    57
  Total in raw dir:    101

── No video found (10) ──
  ✗ 2017 Strade Bianche
  ✗ 2017 Tour de Pologne Stage 1
  ✗ 2017 Tour de Pologne Stage 2
  ✗ 2017 Tour de Pologne Stage 3
  ✗ 2017 Tour de Pologne Stage 4
  ✗ 2017 Tour de Pologne Stage 5
  ✗ 2017 Tour de Pologne Stage 6
  ✗ 2017 Tour de Pologne Stage 7
  ✗ 2017 BinckBank Tour Stage 2
  ✗ 2017 BinckBank Tour Stage 3

── To manu

In [23]:
# Run this to get your TODO list
DISCOVERY_DIR = Path("../data/commentary/discovered")
RAW_DIR       = Path("../data/commentary/raw")

# Races not yet fetched
pending = []
for path in sorted(RAW_DIR.glob("*.json")):
    raw_path = RAW_DIR / path.name
    if raw_path.exists():
        with open(path) as f:
            data = json.load(f)
        if data.get("status") == "no_video_found":
            pending.append({
                "label":    data["label"],
                # "title":    data.get("video", {}).get("title", ""),
                "title":    (data.get("video") or {}).get("title", ""),
                # "url":      data.get("video", {}).get("url", ""),
                "url":      (data.get("video") or {}).get("url", ""),
                "safe_name": path.stem
            })

# Export as CSV for easy manual review
pd.DataFrame(pending).to_csv("../data/commentary/pending_transcripts.csv", index=False)
print(f"{len(pending)} races pending transcript fetch")
print("Saved to data/commentary/pending_transcripts.csv")

20 races pending transcript fetch
Saved to data/commentary/pending_transcripts.csv


In [24]:
def batch_fetch_from_ids(
    video_map: dict,          # {safe_name: video_id}
    race_meta: dict,          # {safe_name: {label, race_name, race_date, stage}}
    delay_seconds: float = 60.0
) -> dict:
    """
    Fetch transcripts from a list of known video IDs.
    No API calls — just transcript fetching with generous delays.
    Works with both automated discoveries and manually found IDs.
    """
    success = errors = 0

    for safe_name, video_id in video_map.items():
        out_path = RAW_DIR / f"{safe_name}.json"

        if out_path.exists():
            print(f"  Skip (exists): {safe_name}")
            continue

        meta  = race_meta.get(safe_name, {})
        label = meta.get("label", safe_name)

        print(f"\nFetching: {label}")
        print(f"  Video ID: {video_id}")

        transcript = fetch_transcript(video_id)

        if not transcript["success"]:
            print(f"  ✗ {transcript['error']}")
            errors += 1
            if any(kw in transcript["error"].lower()
                   for kw in ["blocked", "ip", "429"]):
                print(f"\n⚠ IP block detected. Stopping.")
                print(f"  Saved {success} before block.")
                print(f"  Wait 2-4 hours then resume.")
                break
        else:
            result = {
                "label":      label,
                "race_name":  meta.get("race_name", ""),
                "race_date":  meta.get("race_date", ""),
                "stage":      meta.get("stage"),
                "video": {
                    "video_id":    video_id,
                    "title":       meta.get("title", "manually provided"),
                    "url":         f"https://youtube.com/watch?v={video_id}",
                    "channel":     meta.get("channel", "manual"),
                    "published":   meta.get("race_date", ""),
                    "match_score": 999,
                    "all_candidates": [],
                },
                "transcript": {
                    "snippet_count": transcript["snippet_count"],
                    "raw_chars":     transcript["raw_chars"],
                    "clean_chars":   transcript["clean_chars"],
                    "duration_mins": transcript["duration_mins"],
                    "clean_text":    transcript["clean_text"],
                    "preview_start": transcript["preview_start"],
                    "preview_end":   transcript["preview_end"],
                },
                "status": "transcript_saved",
            }
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(result, f, indent=2, ensure_ascii=False)

            print(f"  ✓ {transcript['clean_chars']:,} chars | "
                  f"{transcript['duration_mins']:.1f} min")
            success += 1

        # Add jitter to look human
        import random
        actual_delay = delay_seconds + random.uniform(-10, 15)
        actual_delay = max(30, actual_delay)
        print(f"  Waiting {actual_delay:.0f}s...")
        time.sleep(actual_delay)

    print(f"\nDone: {success} saved, {errors} errors")
    return {"success": success, "errors": errors}

In [29]:
# Build video_map from discovery cache
video_map = {}
race_meta = {}
for path in sorted(RAW_DIR.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    if data.get("status") == "transcript_saved" and data.get("video"):
        video_map[path.stem] = data["video"]["video_id"]
        race_meta[path.stem] = {
            "label":     data["label"],
            "race_name": data["race_name"],
            "race_date": data["race_date"],
            "stage":     data["stage"],
            "title":     data["video"]["title"],
            "channel":   data["video"]["channel"],
        }

print(f"Videos ready to fetch: {len(video_map)}")

Videos ready to fetch: 38


In [ ]:
for file in 

In [ ]:
# Add manually found video IDs
video_map = {}
race_meta = {}
manual_additions = {
    # "2017_paris_roubaix":  "PO683yZBISc",
    "2017_strade_bianche":   "pCTdlonFsxs",
    # "2017_amstel_gold_race": "dlfHFC8he18",
}

# Merge with automated
video_map.update(manual_additions)

# You'll need to add meta for manual ones
for safe_name in manual_additions:
    if safe_name not in race_meta:
        race_meta[safe_name] = {
            "label":     safe_name.replace("_", " ").title(),
            "race_name": "",  # fill in manually if needed
            "race_date": "",
            "stage":     None,
            "title":     "manually provided",
            "channel":   "manual",
        }

batch_fetch_from_ids(video_map, race_meta, delay_seconds=60)


Fetching: 2017 Strade Bianche
  Video ID: pCTdlonFsxs
  ✓ 2,499 chars | 2.5 min
  Waiting 51s...

Done: 1 saved, 0 errors


{'success': 1, 'errors': 0}

: 

In [32]:
# ── Review all failures across all runs ───────────────────
# Run this anytime to see what needs manual attention

no_video_races = []
error_races    = []

for path in sorted(RAW_DIR.glob("*.json")):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    status = data.get("status", "")
    label  = data.get("label", path.stem)
    if status == "no_video_found":
        no_video_races.append(label)
    elif status.startswith("exception") or status.startswith("transcript_error"):
        error_races.append((label, status))

print(f"Total cached files: {len(list(RAW_DIR.glob('*.json')))}")
print(f"No video found:     {len(no_video_races)}")
print(f"Errors:             {len(error_races)}")

if no_video_races:
    print(f"\n── No video found ──")
    for label in no_video_races:
        print(f"  {label}")

if error_races:
    print(f"\n── Errors ──")
    for label, reason in error_races:
        print(f"  {label}")
        print(f"    {reason}")

Total cached files: 62
No video found:     19
Errors:             4

── No video found ──
  2017 Eschborn-Frankfurt 'Rund um den Finanzplatz'
  2017 Giro d'Italia Stage 14
  2017 Giro d'Italia Stage 18
  2017 Giro d'Italia Stage 19
  2017 La Fleche Wallonne
  2017 Liege - Bastogne - Liege
  2017 Paris-Nice Stage 8
  2017 Tirreno-Adriatico Stage 6
  2017 Tour de Romandie Stage 1
  2017 Tour de Romandie Stage 3
  2017 Tour de Romandie Stage 4
  2017 Tour of Oman Stage 5
  2017 Tour of Oman Stage 6
  2017 Volta Ciclista a Catalunya Stage 1
  2017 Volta Ciclista a Catalunya Stage 2
  2017 Volta Ciclista a Catalunya Stage 3
  2017 Volta Ciclista a Catalunya Stage 5
  2017 Volta Ciclista a Catalunya Stage 6
  2017 Volta Ciclista a Catalunya Stage 7

── Errors ──
  2017 Criterium du Dauphine Stage 1
    transcript_error:
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=u3DLEYjKV2Q! This is most likely caused by:

YouTube is blocking requests from your IP. This usu

In [10]:
def manual_process_race(
    video_id: str,
    race_name: str,
    race_date: str,
    stage: int = None,
    verbose: bool = True
) -> dict:
    """
    Manual override — use when automated search fails.
    Provide the YouTube video ID directly (from the URL after ?v=).
    Saves to the same raw directory as the automated pipeline
    so the rest of the system treats it identically.
    No YouTube Data API used — no quota cost.
    """
    label     = f"{str(race_date)[:4]} {race_name}"
    label    += f" Stage {stage}" if stage else ""
    safe_name = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
    out_path  = RAW_DIR / f"{safe_name}.json"

    if verbose:
        print(f"Manual processing: {label}")
        print(f"  Video ID: {video_id}")
        print(f"  Output:   {out_path.name}")

    # Overwrite existing cache — this is intentional for manual overrides
    if out_path.exists():
        if verbose:
            print(f"  Overwriting existing cache ({out_path.name})")

    # Fetch transcript directly — no API key needed
    transcript = fetch_transcript(video_id)

    if not transcript["success"]:
        if verbose:
            print(f"  ✗ Transcript error: {transcript['error']}")
        result = {
            "label":      label,
            "race_name":  race_name,
            "race_date":  str(race_date)[:10],
            "stage":      stage,
            "video": {
                "video_id":    video_id,
                "title":       "manually provided",
                "url":         f"https://youtube.com/watch?v={video_id}",
                "channel":     "manual",
                "published":   str(race_date)[:10],
                "match_score": 999,
                "all_candidates": [],
            },
            "transcript": None,
            "status":     f"transcript_error:{transcript['error']}",
        }
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        return result

    result = {
        "label":      label,
        "race_name":  race_name,
        "race_date":  str(race_date)[:10],
        "stage":      stage,
        "video": {
            "video_id":    video_id,
            "title":       "manually provided",
            "url":         f"https://youtube.com/watch?v={video_id}",
            "channel":     "manual",
            "published":   str(race_date)[:10],
            "match_score": 999,  # manually provided = perfect match
            "all_candidates": [],
        },
        "transcript": {
            "snippet_count": transcript["snippet_count"],
            "raw_chars":     transcript["raw_chars"],
            "clean_chars":   transcript["clean_chars"],
            "duration_mins": transcript["duration_mins"],
            "clean_text":    transcript["clean_text"],
            "preview_start": transcript["preview_start"],
            "preview_end":   transcript["preview_end"],
        },
        "status": "transcript_saved",
    }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    if verbose:
        print(f"  ✓ Transcript saved: "
              f"{transcript['clean_chars']:,} chars | "
              f"{transcript['duration_mins']:.1f} min")

    return result

In [12]:
manual_process_race(
    video_id="dlfHFC8he18",
    race_name="Amstel Gold Race",
    race_date="2017-04-16",
    stage=None
)

Manual processing: 2017 Amstel Gold Race
  Video ID: dlfHFC8he18
  Output:   2017_amstel_gold_race.json
  Overwriting existing cache (2017_amstel_gold_race.json)
  ✗ Transcript error: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=dlfHFC8he18! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be ret

{'label': '2017 Amstel Gold Race',
 'race_name': 'Amstel Gold Race',
 'race_date': '2017-04-16',
 'stage': None,
 'video': {'video_id': 'dlfHFC8he18',
  'title': 'manually provided',
  'url': 'https://youtube.com/watch?v=dlfHFC8he18',
  'channel': 'manual',
  'published': '2017-04-16',
  'match_score': 999,
  'all_candidates': []},
 'transcript': None,
 'status': 'transcript_error:\nCould not retrieve a transcript for the video https://www.youtube.com/watch?v=dlfHFC8he18! This is most likely caused by:\n\nYouTube is blocking requests from your IP. This usually is due to one of the following reasons:\n- You have done too many requests and your IP has been blocked by YouTube\n- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.\n\nWays to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-tr

---
## Part 2 — Claude Tactical Extraction

Reads saved transcripts from `data/commentary/raw/` and extracts structured
tactical events using Claude. Results are saved to `data/commentary/extracted/`.

**Cost control:**
- Only processes files with `status: transcript_saved` that don't already have an extracted version
- Never re-processes an already-extracted race
- Use `max_extractions` to control daily Claude spend
- At ~$0.02 per race, 50 extractions ≈ $1.00


In [ ]:
EXTRACTION_PROMPT = """You are analyzing cycling race commentary for {race_name}.

Extract tactically significant information from this transcript.
Focus only on information useful for pre-race analysis of future similar stages.
Ignore music, crowd noise, and generic excitement phrases.

Return a JSON object:
{{
  "race_summary": "2-3 sentence tactical summary of what happened",
  "decisive_moment": "the single most important tactical event",
  "commentary_quality": "rich|moderate|thin",
  "commentary_quality_reason": "why you rated it this way",
  "events": [
    {{
      "event_type": "attack|breakaway|chase|team_tactic|rider_observation|weather|terrain",
      "riders": ["LASTNAME Firstname"],
      "teams": ["Team Name"],
      "description": "what happened concisely",
      "tactical_significance": "why this matters for future analysis"
    }}
  ],
  "rider_form_signals": [
    {{
      "rider": "LASTNAME Firstname",
      "signal": "positive|negative|neutral",
      "observation": "what commentary reveals about their condition or ability"
    }}
  ],
  "terrain_observations": [
    "observation about how terrain affected racing dynamics"
  ],
  "usable_for_rag": true,
  "usable_reason": "why this transcript is or isn't useful for RAG"
}}

Return only valid JSON, no other text.

Transcript:
{transcript}"""


def extract_tactical_events(label: str, transcript_text: str) -> dict:
    """
    Run Claude extraction on one transcript.
    Uses first 4000 + last 4000 chars — most tactically dense sections.
    """
    max_chars = 8000
    if len(transcript_text) > max_chars:
        half = max_chars // 2
        text = (transcript_text[:half] +
                "\n[...middle section omitted...]\n" +
                transcript_text[-half:])
    else:
        text = transcript_text

    response = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=3000,
        messages=[{
            "role": "user",
            "content": EXTRACTION_PROMPT.format(
                race_name=label,
                transcript=text
            )
        }]
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"error": "json_parse_failed", "raw": raw[:300]}


print("Part 2 extraction functions ready")


In [ ]:
def run_extraction(
    max_extractions: int = 50,
    verbose: bool = True
) -> dict:
    """
    Run Part 2 Claude extraction.
    - Only processes transcript_saved files not yet extracted
    - Never re-processes already-extracted races
    - max_extractions controls daily Claude spend
      (50 extractions ≈ $1.00 at current pricing)
    """
    raw_files   = sorted(RAW_DIR.glob("*.json"))
    processed   = 0
    skipped     = 0
    success     = 0
    errors      = 0
    total_chars = 0

    print(f"Raw transcripts available: {len(raw_files)}")
    print(f"Max extractions this run:  {max_extractions}")
    print(f"{'─'*55}")

    for raw_path in raw_files:
        if processed >= max_extractions:
            print(f"\nReached max_extractions limit ({max_extractions}).")
            print(f"Re-run to continue with remaining transcripts.")
            break

        # Check if already extracted — never re-charge
        extracted_path = EXTRACTED_DIR / raw_path.name
        if extracted_path.exists():
            skipped += 1
            continue

        # Load raw file
        try:
            with open(raw_path, encoding="utf-8") as f:
                raw_data = json.load(f)
        except Exception as e:
            print(f"  ✗ Could not read {raw_path.name}: {e}")
            errors += 1
            processed += 1
            continue

        # Only extract from successful transcripts
        if raw_data.get("status") != "transcript_saved":
            skipped += 1
            continue

        transcript_text = raw_data.get("transcript", {}).get("clean_text", "")
        if not transcript_text.strip():
            skipped += 1
            continue

        label = raw_data.get("label", raw_path.stem)

        if verbose:
            chars = len(transcript_text)
            est_cost = min(chars, 8000) / 4 / 1_000_000 * 3
            print(f"Extracting: {label}")
            print(f"  Chars: {chars:,} | Est cost: ${est_cost:.4f}")

        try:
            events = extract_tactical_events(label, transcript_text)

            # Build output record
            output = {
                "label":      label,
                "race_name":  raw_data.get("race_name"),
                "race_date":  raw_data.get("race_date"),
                "stage":      raw_data.get("stage"),
                "video_title": raw_data.get("video", {}).get("title"),
                "video_url":  raw_data.get("video", {}).get("url"),
                "channel":    raw_data.get("video", {}).get("channel"),
                "extraction": events,
                "extracted_at": str(datetime.datetime.utcnow()),
            }

            with open(extracted_path, "w", encoding="utf-8") as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            total_chars += min(len(transcript_text), 8000)
            success += 1

            if verbose:
                quality = events.get("commentary_quality", "unknown")
                n_events = len(events.get("events", []))
                print(f"  ✓ Quality: {quality} | Events: {n_events}")

        except Exception as e:
            print(f"  ✗ Extraction failed: {e}")
            errors += 1

        processed += 1
        time.sleep(0.5)

    total_cost = total_chars / 4 / 1_000_000 * 3
    print(f"\n{'─'*55}")
    print(f"Extraction run complete")
    print(f"  Extracted:          {success}")
    print(f"  Skipped (done):     {skipped}")
    print(f"  Errors:             {errors}")
    print(f"  Est. Claude cost:   ${total_cost:.4f}")
    print(f"  Total extracted:    {len(list(EXTRACTED_DIR.glob('*.json')))}")
    return {"success": success, "skipped": skipped, "errors": errors, "cost": total_cost}


print("run_extraction ready")


In [ ]:
# ── PART 2: RUN THIS CELL WHEN READY TO EXTRACT ─────────
# Only runs on transcripts not yet extracted
# max_extractions=50 ≈ $1.00 at current Claude pricing
# Increase or set to None to process all available transcripts

extraction_stats = run_extraction(
    max_extractions=50,
    verbose=True,
)


In [ ]:
# ── Commentary retrieval with graceful fallback ───────────
# Used by the agent layer — always returns something useful

def get_commentary_context(
    race_name: str,
    race_date: str,
    stage: int = None,
    max_chars: int = 4000
) -> str:
    """
    Returns formatted context string for the LLM.
    Checks extracted/ first, falls back to raw transcript,
    falls back to a clear 'no commentary' message.
    Never raises.
    """
    label     = f"{str(race_date)[:4]} {race_name}"
    label    += f" Stage {stage}" if stage else ""
    safe_name = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")

    extracted_path = EXTRACTED_DIR / f"{safe_name}.json"
    raw_path       = RAW_DIR       / f"{safe_name}.json"

    # ── Option 1: extracted tactical events exist ────────────
    if extracted_path.exists():
        try:
            with open(extracted_path, encoding="utf-8") as f:
                data = json.load(f)
            ext = data.get("extraction", {})
            if "error" not in ext:
                parts = []
                parts.append(
                    f"[COMMENTARY: {data.get('channel')} | "
                    f"{data.get('video_title', '')[:55]}]"
                )
                if ext.get("race_summary"):
                    parts.append(f"Summary: {ext['race_summary']}")
                if ext.get("decisive_moment"):
                    parts.append(f"Decisive moment: {ext['decisive_moment']}")
                for e in ext.get("events", [])[:5]:
                    riders = ", ".join(e.get("riders", []))
                    parts.append(
                        f"- [{e['event_type']}] {e['description']}"
                        + (f" ({riders})" if riders else "")
                    )
                for s in ext.get("rider_form_signals", [])[:3]:
                    parts.append(
                        f"- Form signal: {s['rider']} ({s['signal']}): "
                        f"{s['observation']}"
                    )
                return "\n".join(parts)
        except Exception:
            pass

    # ── Option 2: raw transcript exists but not yet extracted ─
    if raw_path.exists():
        try:
            with open(raw_path, encoding="utf-8") as f:
                data = json.load(f)
            if data.get("status") == "transcript_saved":
                text = data.get("transcript", {}).get("clean_text", "")
                if text:
                    if len(text) > max_chars:
                        half = max_chars // 2
                        text = text[:half] + "\n[...]\n" + text[-half:]
                    channel = data.get("video", {}).get("channel", "unknown")
                    title   = data.get("video", {}).get("title", "")[:55]
                    return (
                        f"[RAW COMMENTARY (not yet extracted): "
                        f"{channel} | {title}]\n\n{text}"
                    )
        except Exception:
            pass

    # ── Option 3: nothing available ──────────────────────────
    return (
        f"[NO COMMENTARY AVAILABLE for {label}] "
        f"Analysis is based on structured race data only."
    )


# Quick test
print("=== Commentary retrieval test ===\n")
test_cases = [
    ("Tour de France",    "2023-07-19", 17),   # should have raw
    ("Paris-Roubaix",     "2023-04-09", None), # should have raw
    ("Giro d'Italia",     "2023-05-27", 19),   # not yet ingested
]
for race, date, stage in test_cases:
    ctx = get_commentary_context(race, date, stage, max_chars=300)
    print(f"--- {date[:4]} {race}" + (f" Stage {stage}" if stage else "") + " ---")
    print(ctx[:400])
    print()
